# Example 1.2.1 / 4: Default Values

Companion notebook for **1.2.1 Restrictions, Defaults and Checks** in the AMPL lab Documentation, section "Default Values". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Default Values

> **Note**
> `display rate;` lists only the values the data gave. To see every value, defaults included,
> display it over its indexing set: `display {p in PROD} rate[p];`. Once a default has been
> used, AMPL keeps it as that instance's value: after the second `display` above,
> `display rate;` and `to_pandas()` list `bands` and `plate` too.

A default must satisfy the same restrictions as the data: with `param rate {PROD} > 0 default 0;`,
a product the data leaves out fails `> 0` as soon as its value is used.

The predefined parameter `Infinity` is a useful default for optional bounds. A bound equal to
`Infinity` or `-Infinity` counts as no bound at all. `ampl.get_data("V.ub")` returns the
variables' upper bounds, with `Infinity` shown as Python's `inf`, and
`ampl.var["V"]["a2"].ub()` returns one of them:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set A;
param ub {A} default Infinity;
var V {i in A} >= 0, <= ub[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["A"] = ["a1", "a2"]
ampl.param["ub"] = {"a1": 50}

In [5]:
# run.py
print(ampl.get_data("V.ub").to_pandas())
print(ampl.var["V"]["a2"].ub())

    V.ub
a1  50.0
a2   inf
inf


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set A := a1 a2;
param ub := a1 50;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display V.ub;

V.ub [*] :=
a1        50
a2  Infinity
;

